# Segmentos de precio de departamentos en CABA
**Los Outliers**

**ASIG00202 - Programación Avanzada para Ciencia de Datos - 2/2026**  
**Profesor:** Juan Carlos Cifuentes Durán

**Integrantes:** Alejandra Beatriz Bertran · Ariel Abal · Sebastián Ezequiel Gines

Clasificación binaria de publicaciones: **segmento alto** (1) frente a **estándar** (0).
La lógica vive en módulos Python para que este documento pueda ejecutarse de principio a fin.

**Pregunta:** ¿hasta qué punto superficie, ambientes y barrio permiten identificar publicaciones del segmento de precio alto en la muestra de CABA de 2020?

## 1. Parámetros para repetir el trabajo
La celda `parameters` permite que Papermill ejecute el notebook con opciones elegidas. `run_id` nombra la carpeta de resultados; repetir ese nombre reemplaza sus archivos. La semilla mantiene la misma división de datos. El cuantil decide dónde comienza el segmento alto: cambiarlo cambia la pregunta del trabajo, no mejora por sí solo el modelo.

In [ ]:
archivo = "datasets/departamentos_2020.csv"
salida = "resultados"
run_id = "departamentos_2020"
cuantil = 0.75
semilla = 42
folds = 3

In [ ]:
from pathlib import Path
import json
import sqlite3

import pandas as pd
from IPython.display import Image, Markdown, display
from inmuebles.datos import cargar_y_limpiar, separar_y_etiquetar
from inmuebles.flujo import Configuracion, ejecutar

assert Path(archivo).is_file(), "Ejecutar desde la raíz del proyecto"
config = Configuracion(archivo, salida, run_id, cuantil, semilla, folds)
carpeta = Path(salida) / run_id

## 2. Fuente y preparación
Fuente: [BA Data — Departamentos en venta](https://data.buenosaires.gob.ar/dataset/departamentos-venta). Archivo original de **2020**, descargado el 30/10/2026 y verificado el 05/11/2026.

Cada fila es **un aviso**, no una venta ni necesariamente una propiedad diferente. Nota exploratoria: el archivo de 2020 no incluye antigüedad.

Quitamos repeticiones exactas y avisos con precios inválidos. Las superficies fuera de 10–1000 m² y los ambientes fuera de 1–20 se dejan como datos desconocidos: podrían ser errores, aunque algunos casos extremos podrían ser reales. Unificamos la escritura de barrios y direcciones. No calculamos superficies a partir del precio, porque introduciríamos la respuesta en los datos para predecir. Después, el Pipeline aprende cómo completar los faltantes usando sólo los datos de entrenamiento.

In [ ]:
datos, auditoria = cargar_y_limpiar(archivo)
display(pd.Series(auditoria, name="Cantidad").to_frame())
display(datos[["superficie", "ambientes", "precio_usd"]].describe())

## 3. Datos para aprender y datos para evaluar
Reservamos el 20 % de las **direcciones normalizadas** para la evaluación final (`GroupShuffleSplit`). Todos los avisos de una misma dirección quedan juntos, aunque el porcentaje de filas no sea exactamente 20 %. Distintas escrituras de una dirección pueden escapar a esta agrupación.

Con los datos de entrenamiento calculamos el precio que deja aproximadamente al 25 % más caro por encima: es el **percentil 75**, USD 235.000. Un aviso es **alto** si supera ese valor y **estándar** si es menor o igual. Es una categoría de precio, no una medida de lujo.

En la notación vista en clase, **X** son superficie, ambientes y barrio; **y** es la categoría alto o estándar. Durante el entrenamiento damos ejemplos con su categoría correcta. Para predecir una categoría sólo damos las tres características: incluir el precio sería dar la respuesta. Tampoco entran pesos, precio/m², cotización, dirección ni la categoría en los datos de entrada.

In [ ]:
train, test, umbral = separar_y_etiquetar(datos, cuantil, semilla)
assert set(train.grupo_direccion).isdisjoint(set(test.grupo_direccion))
assert umbral == train.precio_usd.quantile(cuantil)
display(Markdown(f"**Umbral calculado en entrenamiento: USD {umbral:,.0f}.**"))
display(pd.DataFrame({
    "Partición": ["Entrenamiento", "Prueba"],
    "Publicaciones": [len(train), len(test)],
    "Direcciones": [train.grupo_direccion.nunique(), test.grupo_direccion.nunique()],
    "Proporción de segmento alto": [train.segmento_alto.mean(), test.segmento_alto.mean()],
}))

## 4. Preparar, entrenar y comparar
Como vimos con las transformaciones componibles, cada paso tiene una tarea y se une al siguiente. `ColumnTransformer` prepara cada tipo de dato: completa números faltantes con la mediana, los lleva a una escala común (`StandardScaler`) y convierte los barrios en columnas de ceros y unos (`OneHotEncoder`). Los barrios desconocidos se pueden procesar sin detener la predicción.

`Pipeline` une la preparación y el modelo. En **fit** aprende los ajustes con entrenamiento; en **predict** aplica esos mismos ajustes para predecir. No aprende de los datos reservados para la evaluación: así evitamos la **fuga de información**, explicada en la materia.

Comparamos una referencia que siempre responde estándar (`DummyClassifier`), **regresión logística**, que combina las características para estimar la categoría, y **Random Forest**, que reúne las decisiones de 120 árboles.

Probamos varias configuraciones, llamadas **hiperparámetros**. Para logística: C de 0.1, 1 o 10 y pesos de clases normales o balanceados. Para Random Forest: profundidad 12 o sin límite y al menos 5 o 20 ejemplos por hoja.

`GridSearchCV` prueba esas combinaciones con **validación cruzada**: divide entrenamiento en tres grupos; en cada ronda aprende con dos y se revisa con el restante. Todos usan las mismas divisiones y una dirección permanece en un solo grupo (`StratifiedGroupKFold`). Se intenta mantener la proporción de categorías, hasta donde lo permite la agrupación. Elegimos el mejor F1 promedio **antes** de abrir la evaluación final.

In [ ]:
resumen = ejecutar(config)
assert resumen["grupos_compartidos"] == 0
comparacion = pd.read_csv(carpeta / "metricas.csv")
display(comparacion.drop(columns="parametros").round(4))

## 5. Evaluación final: cuatro preguntas
Como en el ejemplo de clasificación de la materia:

- **Accuracy:** ¿qué proporción del total clasificamos bien?
- **Precision:** de los avisos marcados como altos, ¿cuántos eran altos de verdad?
- **Recall:** de todos los altos reales, ¿cuántos encontramos?
- **F1:** ¿cómo combinamos Precision y Recall? Es su media armónica y baja si una de las dos es baja.

La **matriz de confusión** muestra aciertos y errores: las filas indican la categoría real y las columnas, la predicción; el orden es estándar y alto. Permite distinguir avisos estándar marcados como altos (falsos positivos) de altos que no detectamos (falsos negativos).

La referencia acierta cerca del 75 % porque siempre responde estándar, pero no detecta ningún alto y su F1 es cero. Por eso no elegimos sólo por Accuracy.

Evaluamos los tres modelos con los datos reservados, aunque el ganador ya estaba elegido por validación cruzada. Sus promedios de validación pueden ser algo optimistas por haberlos usado para elegir. La variación entre rondas no es un intervalo de confianza. El límite de probabilidad se mantiene en 0.5 y no se modifica a partir de prueba.

In [ ]:
ganador = resumen["configuracion"]["modelo_elegido"]
fila = comparacion.set_index("modelo").loc[ganador]
base = comparacion.set_index("modelo").loc["base"]
display(Markdown(
    f"**Elegido por CV: {ganador}.** F1 CV: {fila.f1_cv_mean:.3f} "
    f"(desviación {fila.f1_cv_std:.3f}). F1 de prueba: {fila.f1_test:.3f}. "
    f"Recall de prueba: {fila.recall_test:.3f}. Accuracy base: {base.accuracy_test:.3f}."
))
for nombre in ["03_comparacion.png", "04_matrices_confusion.png"]:
    display(Image(filename=str(carpeta / "graficos" / nombre)))

## 6. Explorar e interpretar
Los gráficos del notebook usan los datos de **entrenamiento**. El diagrama de puntos ayuda a ver superficie y precio; las barras permiten comparar barrios y modelos; los histogramas muestran cómo se distribuyen los valores. Para que se lean bien, el gráfico de puntos muestra una muestra; el entrenamiento usa todos los datos que le corresponden.

Los coeficientes de la logística y las importancias de Random Forest describen relaciones que usa cada modelo para predecir. No demuestran que una característica cause un precio.

In [ ]:
for nombre in ["01_distribuciones.png", "02_exploracion.png", "05_variables.png"]:
    display(Image(filename=str(carpeta / "graficos" / nombre)))

## 7. Guardar los datos y resultados
**SQLite** es una base relacional guardada en un archivo. Organizamos la información en cinco tablas: `datos_limpios` guarda los avisos y a qué grupo pertenecen; `datos_preprocesados`, los números preparados para el modelo; `predicciones`, lo que respondió cada modelo; `metricas`, su desempeño; y `configuracion`, las opciones usadas. `fila_fuente` permite relacionar una predicción con su aviso original. La preparación se aprende sólo con entrenamiento.

También preparamos tres **colecciones de MongoDB**. En una base de documentos podemos guardar juntos un aviso limpio y sus valores preparados dentro de `datos_entrada`. `resultados_modelo` reúne predicciones y métricas globales, y `configuracion` guarda las opciones del trabajo. La importación a tu cuenta es manual y sigue pendiente; los archivos preparados no prueban que se hayan cargado.

In [ ]:
with sqlite3.connect(carpeta / "proyecto.sqlite") as conexion:
    tablas = [r[0] for r in conexion.execute(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
    )]
    conteos = {tabla: conexion.execute(
        f'SELECT count(*) FROM "{tabla}"'
    ).fetchone()[0] for tabla in tablas}
    display(pd.Series(conteos, name="Filas").to_frame())
    display(pd.read_sql_query("SELECT * FROM metricas", conexion))
    huerfanas = conexion.execute(
        "SELECT count(*) FROM predicciones p LEFT JOIN datos_limpios d "
        "ON p.fila_fuente=d.fila_fuente WHERE d.fila_fuente IS NULL"
    ).fetchone()[0]
    assert huerfanas == 0
assert conteos["datos_limpios"] == resumen["limpieza"]["filas_limpias"]
assert conteos["datos_preprocesados"] == conteos["datos_limpios"]
assert conteos["predicciones"] == 3 * resumen["prueba"]["filas"]

### Archivos para MongoDB
Este paso genera archivos JSON con un documento por línea, listos para importar con Compass. Para ahorrar espacio, de los valores preparados sólo se guardan los que no son cero; los demás valen cero. La configuración conserva la lista y el orden completos. La exportación es local y no se conecta a tu cuenta.

In [ ]:
from preparar_mongodb import preparar

paquete_mongo = preparar()
display(pd.DataFrame(paquete_mongo["colecciones"]).T[["archivo", "documentos", "bytes"]])
display(Markdown("**Estado: archivos preparados. Importación remota pendiente del estudiante.**"))

## 8. Repetir el análisis y reconocer sus límites
Como vimos con entornos reproducibles, logging y Papermill, guardamos las opciones, versiones y el registro de lo ocurrido. Bajo el mismo `run_id` quedan modelo, tablas, gráficos y resultados. Conservamos el modelo aprendido con entrenamiento, sin volver a enseñarle con los datos de prueba.

Estos resultados describen **avisos de 2020**, concentrados en el segundo trimestre. La categoría alto es relativa al precio de esa muestra. Faltan características relevantes, como estado y servicios del edificio. Hay errores y avisos repetidos; unificar direcciones ayuda, pero no identifica perfectamente cada inmueble.

Para evaluar tasaciones actuales harían falta datos recientes, una identificación más confiable de propiedades y una prueba con otro período. No cambiamos variables ni opciones del modelo mirando la evaluación final.

In [ ]:
display(pd.Series(resumen["configuracion"]["versiones"], name="Versión").to_frame())
display(Markdown(f"Archivos de esta corrida: `{carpeta}`. "
                 f"Tiempo de ejecución: {resumen['duracion_segundos']:.1f} segundos."))